# Week 11 — Deployment
### *Turning a notebook demo into a system people can actually use (without it falling over).*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week11_deployment.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Learning objectives (by the end of this notebook, you can…)
- **Explain** the difference between a notebook demo and a deployed app (UX + reliability + security).
- **Define** latency, throughput, and concurrency in plain language.
- **Identify** common deployment failure modes: timeouts, rate limits, bad inputs, missing secrets.
- **Implement** 3 “deployment guardrails” in a small app: input validation, timeouts, and safe fallbacks.
- **Add** lightweight observability to an app: request logs + basic metrics.
- **Deploy** a Gradio app (Colab share) and describe what it would take to ship on HF Spaces.

---

## How Week 11 fits into our course arc
Week 6–10 built the system “inside the notebook”:
- RAG + agents + safety + evaluation + monitoring

Week 11 asks:
> **How do we ship it?**  
> (and how do we keep it reliable when users do weird things?)

Week 12 then asks:
> How do we improve it (prompting vs RAG vs fine-tuning)?

---

# Week structure (one content day + one test day)
- **Day 1 (content):** deployment basics + live demo + lab kickoff
- **Day 2 (test):** course assessment / review  
  (optional: finish lab in-class if time remains)


In [1]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys, platform, time, random
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/content/main")
from course_utils import show_mermaid, lab11_setup

lab11_setup()
random.seed(11)
np.random.seed(11)
print(f"✅ Environment ready! Python {platform.python_version()}")


installing mermaid-python
Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
installing dspy
✅ Environment ready! Python 3.12.12


---

# Day 1 — Deployment basics (content day)

## **Motivating success**
In a notebook, your system works:
- you run one cell
- you test one question
- you read the output

## **Motivating failure**
In the real world, users do things like:
- paste 10,000 characters
- click “Submit” 20 times
- ask for disallowed content
- hit the system at the exact moment the API is slow
- break your assumptions

Deployment is about:
> *making “works on my laptop” into “works for users.”*

---

## **Guiding questions**
- What does “fast enough” mean?
- What happens when the model or tool is slow?
- How do we handle bad inputs safely?
- How do we keep secrets safe?
- How do we observe real failures after shipping?


In [4]:
# @title Unifying Diagram v8 (Mermaid): Deployment layer added
mermaid = """
flowchart LR
  U[User] --> UI["UI (Web / Gradio)"]
  UI --> API["App Server / Router"]
  API --> POL["Policy + Guardrails"]
  POL --> PIPE["AI Pipeline (RAG / Agent)"]
  PIPE --> LLM["LLM Provider"]
  PIPE --> VDB["(Vector / Memory DB)"]
  PIPE --> TOOLS["Tools/APIs"]
  API --> LOGS["(Logs & Metrics)"]
  LOGS --> MON["Dashboards / Alerts"]

  classDef dep fill:#eef,stroke:#335,color:#000;
  classDef obs fill:#fff7e6,stroke:#b7791f,color:#000;
  class UI,API dep;
  class LOGS,MON obs;
"""
show_mermaid(mermaid)


installing mermaid-python


---

## **A practical definition of latency and throughput**
### Latency
“How long does one request take?”

- If a student clicks “Submit,” latency is the wait time until the answer appears.
- For AI, latency often includes retrieval + generation + tool calls + network.

### Throughput
“How many requests can we handle per second/minute?”

A system can have:
- low latency for one user
- but collapse when 50 users arrive

### Concurrency
“How many requests are in flight at once?”

---

### Reflection
> If your system takes 10 seconds per request, is it “bad”?  
> What if it’s a homework helper used once per day? What if it’s customer chat?


In [5]:
# @title Demo: measuring latency in Python
import time

def fake_model_call(tokens=400, ms_per_token=2.0):
    time.sleep(tokens * ms_per_token / 1000.0)
    return "done"

t0 = time.perf_counter()
_ = fake_model_call(tokens=200, ms_per_token=2.0)
dt = time.perf_counter() - t0
print("Latency (s):", round(dt, 3))


Latency (s): 0.4


---

## **Common deployment failure modes (and what to do)**
### 1) Missing secrets / config
- Mitigation: check env vars at startup; show friendly UI error.

### 2) Rate limits
- Mitigation: retries with backoff, queueing, caching (carefully).

### 3) Timeouts
- Mitigation: time budgets; fall back to a simpler response.

### 4) Bad inputs
- Mitigation: input validation (length), policy checks, refusal/abstain.

### 5) Hidden regressions
- Mitigation: canary eval (Week 10) + logging dashboards.


In [6]:
# @title Demo: safe secrets pattern (environment variables)
import os
key = os.getenv("OPENAI_API_KEY")
if key is None:
    print("⚠️ No OPENAI_API_KEY set. In production: show friendly UI error; avoid stack traces.")
else:
    print("✅ Key is set (do NOT print it). Length:", len(key))


✅ Key is set (do NOT print it). Length: 51


In [7]:
# @title Demo: minimal input validation
def validate_user_text(text: str, max_chars: int = 1500) -> str:
    text = (text or "").strip()
    if len(text) == 0:
        return "ERROR: empty input"
    if len(text) > max_chars:
        return f"ERROR: input too long ({len(text)} chars). Please shorten."
    return "OK"

for t in ["", "hello", "x"*2000]:
    print(validate_user_text(t))


ERROR: empty input
OK
ERROR: input too long (2000 chars). Please shorten.


---

## **Caching: the easiest latency win (sometimes)**
Caching means: if we see the same request again, reuse the old answer.

Pros: cheaper + faster.  
Cons: privacy risks + stale answers.

A safe teaching default:
- cache only “public” questions, or cache briefly.


In [8]:
# @title Demo: caching reduces repeated latency (toy)
import time
_cache = {}

def cached_call(prompt: str):
    if prompt in _cache:
        return _cache[prompt], True
    time.sleep(0.6)
    ans = f"Answer for: {prompt[:30]}..."
    _cache[prompt] = ans
    return ans, False

for i in range(3):
    t0 = time.perf_counter()
    ans, hit = cached_call("What is RAG?")
    dt = time.perf_counter() - t0
    print(f"run {i}: cache_hit={hit}  latency={dt:.3f}s")


run 0: cache_hit=False  latency=0.600s
run 1: cache_hit=True  latency=0.000s
run 2: cache_hit=True  latency=0.000s


### Reflection
> What is a scenario where caching could be dangerous?  
> Think: personal data, personalization, and rapidly changing info.


---

## **Observability in a deployed app**
Minimum viable observability:
- request log table (timestamp, input length, latency, status)
- simple metric: average latency, % errors


In [9]:
# @title Demo: a minimal request log table
import pandas as pd, time
request_logs = []

def log_request(user_text: str, latency_s: float, cache_hit: bool, status: str):
    request_logs.append({
        "t": time.strftime("%H:%M:%S"),
        "len": len(user_text or ""),
        "latency_s": round(latency_s, 3),
        "cache_hit": cache_hit,
        "status": status,
    })

log_request("What is RAG?", 0.62, False, "ok")
log_request("What is RAG?", 0.01, True, "ok")
log_request("", 0.00, False, "error")

pd.DataFrame(request_logs)


,t,len,latency_s,cache_hit,status
0,15:47:58,12,0.62,False,ok
1,15:47:58,12,0.01,True,ok
2,15:47:58,0,0.00,False,error


---

## **Gradio and deployment**
For this course, we’ll use **Gradio** as our primary “deployment surface”:
- fast to build
- easy to share from Colab
- easy to export to HF Spaces later

Mental model:
> Gradio = a tiny web server + UI that calls your Python function.

In Lab 11 you’ll build:
- a handler function with validation + caching + logging
- a UI that calls the handler
- a deployment checklist


---

<details>
<summary><strong>Instructor Notes</strong></summary>

### Suggested pacing (Day 1: 50 min)
- 0–8: motivating success/failure
- 8–15: unifying diagram v8
- 15–25: latency/throughput/concurrency + timing demo
- 25–35: failure modes (secrets, rate limits, timeouts, inputs)
- 35–42: caching demo
- 42–50: observability minimum + lab kickoff

### Test day option
If you have 10–15 minutes after the test, have students launch their Gradio apps and share links.

</details>
